# Air Quality Forecasting - 48 Hour Prediction

In [ ]:
import pandas as pd
import numpy as np
from statsmodels.tsa.statespace.sarimax import SARIMAX

# Load data
df = pd.read_csv("AirQualityUCI_clean.csv")

# Combine date and time
df['Datetime'] = pd.to_datetime(df['Date'].astype(str).str.strip() + ' ' + df['Time'].astype(str).str.strip(), dayfirst=True, errors='coerce')
df.set_index('Datetime', inplace=True)
df.drop(columns=['Date', 'Time'], inplace=True)

# Replace -200 with np.nan
for col in df.select_dtypes(include='number').columns:
    df[col] = np.where(df[col] == -200, np.nan, df[col])

# Fill missing values
df = df.fillna(method='ffill').fillna(method='bfill')


## Forecasting the Next 48 Hours

In [ ]:
# Forecast columns
forecast_columns = [
    "CO(GT)", "PT08.S1(CO)", "NMHC(GT)", "C6H6(GT)", "PT08.S2(NMHC)",
    "NOx(GT)", "PT08.S3(NOx)", "NO2(GT)", "PT08.S4(NO2)",
    "PT08.S5(O3)", "T", "RH", "AH"
]

# Forecast steps
forecast_steps = 48
forecast_results = {}

for col in forecast_columns:
    try:
        model = SARIMAX(df[col], order=(1, 1, 1), enforce_stationarity=False, enforce_invertibility=False)
        results = model.fit(disp=False)
        forecast = results.forecast(steps=forecast_steps)
        forecast_results[col] = forecast
    except:
        forecast_results[col] = [np.nan] * forecast_steps

# Compile forecast
forecast_df = pd.DataFrame(forecast_results)
forecast_df.index = pd.date_range(start=df.index[-1] + pd.Timedelta(hours=1), periods=48, freq='H')
forecast_df.to_excel("air_quality_48hr_forecast_submission.xlsx")


## RMSE Validation and Model Comparison

In [ ]:

# SARIMA RMSE Validation on Last 10% of Data
from sklearn.metrics import mean_squared_error

rmse_results = {}
n = len(df)
test_size = int(n * 0.1)
train_df = df[:-test_size]
test_df = df[-test_size:]

for col in forecast_columns:
    try:
        model = SARIMAX(train_df[col], order=(1, 1, 1), enforce_stationarity=False, enforce_invertibility=False)
        results = model.fit(disp=False)
        pred = results.forecast(steps=test_size)
        rmse = mean_squared_error(test_df[col].dropna(), pred[:len(test_df[col].dropna())], squared=False)
        rmse_results[col] = rmse
    except:
        rmse_results[col] = np.nan

rmse_df = pd.DataFrame.from_dict(rmse_results, orient='index', columns=['SARIMA_RMSE'])
rmse_df.sort_values(by='SARIMA_RMSE', ascending=False)


In [ ]:

# Prophet Forecasting and RMSE Comparison

from prophet import Prophet
from sklearn.metrics import mean_squared_error

prophet_forecasts = {}
prophet_rmse_results = {}

for col in forecast_columns:
    try:
        df_prophet = df[[col]].reset_index().rename(columns={'Datetime': 'ds', col: 'y'})
        df_prophet.dropna(inplace=True)

        model = Prophet()
        model.fit(df_prophet)

        future = model.make_future_dataframe(periods=48, freq='H')
        forecast = model.predict(future)

        # Save forecast
        yhat = forecast[['ds', 'yhat']].tail(48).set_index('ds')
        prophet_forecasts[col] = yhat['yhat']

        # RMSE on last 10% of data
        train_prophet = df_prophet[:-test_size]
        test_prophet = df_prophet[-test_size:]
        model_rmse = Prophet().fit(train_prophet)
        future_rmse = model_rmse.make_future_dataframe(periods=test_size, freq='H')
        forecast_rmse = model_rmse.predict(future_rmse).tail(test_size)
        rmse = mean_squared_error(test_prophet['y'].values, forecast_rmse['yhat'].values, squared=False)
        prophet_rmse_results[col] = rmse

    except Exception as e:
        prophet_forecasts[col] = pd.Series([np.nan] * 48)
        prophet_rmse_results[col] = np.nan

# Combine and save forecast
prophet_forecast_df = pd.DataFrame(prophet_forecasts)
prophet_forecast_df.index = pd.date_range(start=df.index[-1] + pd.Timedelta(hours=1), periods=48, freq='H')
prophet_forecast_df.to_excel("air_quality_48hr_forecast_submission_prophet.xlsx")

# Create RMSE DataFrame
prophet_rmse_df = pd.DataFrame.from_dict(prophet_rmse_results, orient='index', columns=['Prophet_RMSE'])
prophet_rmse_df.sort_values(by='Prophet_RMSE', ascending=False)
